# DDRI Full-Station Station-Hour Baseline

161개 공통 스테이션 `station-hour` 데이터셋을 사용해 전체 서비스용 1차 baseline 모델을 학습한다.

- Train: `2023`
- Validation: `2024`
- Final Train: `2023 + 2024`
- Test: `2025`
- Model: `LightGBM_RMSE`

이번 노트북은 대표 대여소 15개 실험과 분리된 전체 스테이션 본실험 트랙이다.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
plt.rcParams['axes.unicode_minus'] = False
import pandas as pd
import seaborn as sns
from lightgbm import LGBMRegressor
sns.set_theme(style='whitegrid', rc={'font.family': 'Apple SD Gothic Neo', 'axes.unicode_minus': False})

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')
SOURCE_DIR = ROOT / '3조 공유폴더/군집별 데이터_전체 스테이션/full_data'
WORK_DIR = ROOT / 'works/06_prediction_long_full'
OUTPUT_DATA_DIR = WORK_DIR / 'output/data'
IMAGE_DIR = WORK_DIR / 'output/images'
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SOURCE_DIR / 'ddri_prediction_long_train_2023_2024.csv'
TEST_PATH = SOURCE_DIR / 'ddri_prediction_long_test_2025.csv'

DTYPE_MAP = {
    'station_id': 'int32',
    'hour': 'int8',
    'rental_count': 'int16',
    'cluster': 'int8',
    'mapped_dong_code': 'int32',
    'weekday': 'int8',
    'month': 'int8',
    'holiday': 'int8',
    'temperature': 'float32',
    'humidity': 'float32',
    'precipitation': 'float32',
    'wind_speed': 'float32',
}

## 1. 데이터 로드

원본 CSV는 공유폴더 `full_data`에 두고, 실험 산출물만 `works/06_prediction_long_full`에 저장한다.

In [3]:
train = pd.read_csv(TRAIN_PATH, dtype=DTYPE_MAP)
test = pd.read_csv(TEST_PATH, dtype=DTYPE_MAP)

for df in [train, test]:
    df['date'] = pd.to_datetime(df['date'])
    df['datetime'] = df['date'] + pd.to_timedelta(df['hour'].astype('int16'), unit='h')

train.shape, test.shape, train['station_id'].nunique(), test['station_id'].nunique()

((2824584, 14), (1410360, 14), 161, 161)

## 2. 시계열 피처 생성

시간대 수요는 직전 1시간, 24시간, 1주 전 패턴의 영향을 강하게 받을 수 있으므로 `lag`와 `rolling` 피처를 추가한다.

주의: `rolling_mean/std`는 반드시 `station_id`별로 계산해 스테이션 경계 누수를 막는다.

In [4]:
combined = pd.concat([train.assign(dataset='train'), test.assign(dataset='test')], ignore_index=True)
combined = combined.sort_values(['station_id', 'datetime']).reset_index(drop=True)

grouped_target = combined.groupby('station_id')['rental_count']
combined['lag_1h'] = grouped_target.shift(1).astype('float32')
combined['lag_24h'] = grouped_target.shift(24).astype('float32')
combined['lag_168h'] = grouped_target.shift(168).astype('float32')
combined['rolling_mean_24h'] = grouped_target.transform(lambda s: s.shift(1).rolling(24).mean()).astype('float32')
combined['rolling_mean_168h'] = grouped_target.transform(lambda s: s.shift(1).rolling(168).mean()).astype('float32')
combined['rolling_std_24h'] = grouped_target.transform(lambda s: s.shift(1).rolling(24).std()).astype('float32')

train_model = combined[combined['dataset'] == 'train'].copy()
test_model = combined[combined['dataset'] == 'test'].copy()

FEATURE_COLUMNS = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_mean_168h', 'rolling_std_24h'
]
CATEGORICAL_COLUMNS = ['station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']

train_model[FEATURE_COLUMNS].head(3)

,station_id,cluster,mapped_dong_code,hour,weekday,month,holiday,temperature,humidity,precipitation,wind_speed,lag_1h,lag_24h,lag_168h,rolling_mean_24h,rolling_mean_168h,rolling_std_24h
0,2301,2,11680510,0,6,1,1,-2.0,81.0,0.0,9.3,NaN,NaN,NaN,NaN,NaN,NaN
1,2301,2,11680510,1,6,1,1,-1.7,82.0,0.0,9.3,0.0,NaN,NaN,NaN,NaN,NaN
2,2301,2,11680510,2,6,1,1,-0.8,77.0,0.0,10.8,0.0,NaN,NaN,NaN,NaN,NaN


## 3. 연 단위 분할

계절성과 날씨 분포를 반영하기 위해 `2023` 학습, `2024` validation, `2025` 테스트 구조를 사용한다.

In [5]:
train_2023 = train_model[train_model['date'] < '2024-01-01'].copy()
valid_2024 = train_model[train_model['date'] >= '2024-01-01'].copy()
full_train = train_model.copy()

X_train = train_2023[FEATURE_COLUMNS].copy()
y_train = train_2023['rental_count'].copy()

X_valid = valid_2024[FEATURE_COLUMNS].copy()
y_valid = valid_2024['rental_count'].copy()

X_full = full_train[FEATURE_COLUMNS].copy()
y_full = full_train['rental_count'].copy()

X_test = test_model[FEATURE_COLUMNS].copy()
y_test = test_model['rental_count'].copy()

for frame in [X_train, X_valid, X_full, X_test]:
    for column in CATEGORICAL_COLUMNS:
        frame[column] = frame[column].astype('category')

train_2023.shape, valid_2024.shape, full_train.shape, test_model.shape

((1410360, 21), (1414224, 21), (2824584, 21), (1410360, 21))

## 4. LightGBM baseline 학습

전체 161개 스테이션 본실험의 첫 모델은 `LightGBM_RMSE` 하나로 시작한다. 추천 모델 전체 비교는 이 baseline 확인 후 확장한다.

In [6]:
def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
    }

model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='regression',
)

results = []

model.fit(X_train, y_train, categorical_feature=CATEGORICAL_COLUMNS)
valid_pred = model.predict(X_valid)
results.append(evaluate_predictions('LightGBM_RMSE_Full', 'validation_2024', y_valid, valid_pred))

model.fit(X_full, y_full, categorical_feature=CATEGORICAL_COLUMNS)
test_pred = model.predict(X_test)
results.append(evaluate_predictions('LightGBM_RMSE_Full', 'test_2025_refit', y_test, test_pred))

results_df = pd.DataFrame(results)
results_df

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037642 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1618
[LightGBM] [Info] Number of data points in the train set: 1410360, number of used features: 17
[LightGBM] [Info] Start training from score 0.688392


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.048872 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1642
[LightGBM] [Info] Number of data points in the train set: 2824584, number of used features: 17
[LightGBM] [Info] Start training from score 0.693763


,model,split,rmse,mae,r2
0,LightGBM_RMSE_Full,validation_2024,0.9735,0.6234,0.4463
1,LightGBM_RMSE_Full,test_2025_refit,0.8624,0.5594,0.4369


## 5. 오류 요약과 feature importance 저장

전체 평균 지표와 함께, 2025 테스트 기준 스테이션별 오류 요약도 같이 저장한다.

In [7]:
metrics_path = OUTPUT_DATA_DIR / 'ddri_station_hour_full_model_metrics.csv'
results_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')

importance_df = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)
feature_label_map = {
    'station_id': '대여소 ID(Station ID)',
    'cluster': '군집(Cluster)',
    'mapped_dong_code': '행정동 코드(Mapped Dong Code)',
    'hour': '시간대(Hour)',
    'weekday': '요일(Weekday)',
    'month': '월(Month)',
    'holiday': '공휴일 여부(Holiday)',
    'temperature': '기온(Temperature)',
    'humidity': '습도(Humidity)',
    'precipitation': '강수량(Precipitation)',
    'wind_speed': '풍속(Wind Speed)',
    'lag_1h': '1시간 전 대여량(Lag 1h)',
    'lag_24h': '24시간 전 대여량(Lag 24h)',
    'lag_168h': '168시간 전 대여량(Lag 168h)',
    'rolling_mean_24h': '24시간 이동평균(Rolling Mean 24h)',
    'rolling_mean_168h': '168시간 이동평균(Rolling Mean 168h)',
    'rolling_std_24h': '24시간 이동표준편차(Rolling Std 24h)',
}
importance_df['feature_kr'] = importance_df['feature'].map(feature_label_map).fillna(importance_df['feature'])
importance_path = OUTPUT_DATA_DIR / 'ddri_station_hour_full_lightgbm_feature_importance.csv'
importance_df.to_csv(importance_path, index=False, encoding='utf-8-sig')

test_eval = test_model[['station_id', 'date', 'hour', 'rental_count']].copy()
test_eval['prediction'] = test_pred
test_eval['abs_error'] = (test_eval['rental_count'] - test_eval['prediction']).abs()
test_eval['squared_error'] = (test_eval['rental_count'] - test_eval['prediction']) ** 2

station_error_df = test_eval.groupby('station_id').agg(
    actual_mean=('rental_count', 'mean'),
    predicted_mean=('prediction', 'mean'),
    mae=('abs_error', 'mean'),
    rmse=('squared_error', lambda s: float(s.mean() ** 0.5)),
).reset_index().sort_values('mae', ascending=False)

station_error_path = OUTPUT_DATA_DIR / 'ddri_station_hour_full_station_error_summary.csv'
station_error_df.to_csv(station_error_path, index=False, encoding='utf-8-sig')

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(12), x='importance', y='feature_kr', hue='feature_kr', legend=False, palette='Blues_r')
plt.title('전체 스테이션 시간대(Hour) LightGBM 피처 중요도')
plt.xlabel('중요도(Importance)')
plt.ylabel('피처(Feature)')
plt.tight_layout()
feature_image_path = IMAGE_DIR / 'ddri_station_hour_full_lightgbm_feature_importance.png'
plt.savefig(feature_image_path, dpi=150)
plt.close()

metrics_path, importance_path, station_error_path, feature_image_path

(PosixPath('/Users/cheng80/Desktop/ddri_work/works/06_prediction_long_full/output/data/ddri_station_hour_full_model_metrics.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/06_prediction_long_full/output/data/ddri_station_hour_full_lightgbm_feature_importance.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/06_prediction_long_full/output/data/ddri_station_hour_full_station_error_summary.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/06_prediction_long_full/output/images/ddri_station_hour_full_lightgbm_feature_importance.png'))

In [8]:
results_df, importance_df.head(12), station_error_df.head(10)

(                model            split    rmse     mae      r2
 0  LightGBM_RMSE_Full  validation_2024  0.9735  0.6234  0.4463
 1  LightGBM_RMSE_Full  test_2025_refit  0.8624  0.5594  0.4369,
               feature  importance                     feature_kr
 0          station_id        2297             대여소 ID(Station ID)
 1                hour        1692                      시간대(Hour)
 2             weekday         563                    요일(Weekday)
 3            lag_168h         556          168시간 전 대여량(Lag 168h)
 4              lag_1h         491              1시간 전 대여량(Lag 1h)
 5         temperature         488                기온(Temperature)
 6             lag_24h         414            24시간 전 대여량(Lag 24h)
 7   rolling_mean_168h         331  168시간 이동평균(Rolling Mean 168h)
 8            humidity         314                   습도(Humidity)
 9    rolling_mean_24h         305    24시간 이동평균(Rolling Mean 24h)
 10      precipitation         294             강수량(Precipitation)
 11            